### Sprint 9

In [2]:
import re
import numpy as np
import pyperclip
from collections import Counter

---
## Nivell 1

### Exercici 1 — Calculadora de l'índex de massa corporal (IMC)

El programa calcula l'IMC a partir del pes i l'alçada introduïts per l'usuari i el classifica en les categories establertes per l'OMS. S'hi inclou una validació prèvia per garantir que els valors siguin numèrics i coherents.
He fet quatre funcions perquè es part de les bones pràctiques que així sigui.

In [3]:
def validar_dades(pes, alcada):
    """
    Valida que el pes i l'alçada siguin numèrics i dins de rangs raonables.
    pes: float — en quilograms
    alcada: float — en metres
    """
    if not isinstance(pes, (int, float)) or not isinstance(alcada, (int, float)):
        return False, "Les dades han de ser numèriques."
    if pes <= 0 or alcada <= 0:
        return False, "El pes i l'alçada han de ser positius."
    if pes > 300 or alcada > 2.5:
        return False, "Valors fora de rang raonable (pes > 300 kg o alçada > 2.5 m)."
    return True, ""


def calcular_imc(pes, alcada):
    return pes / (alcada ** 2)


def classificar_imc(imc):
    if imc < 18.5:
        return "Baix pes"
    elif imc < 25:
        return "Pes normal"
    elif imc < 30:
        return "Sobrepès"
    else:
        return "Obesitat"


def imc_usuari():
    pes_input = input("Introdueix el teu pes (kg): ").replace(",", ".")
    alcada_input = input("Introdueix la teva alçada (m): ").replace(",", ".")

    try:
        pes = float(pes_input)
        alcada = float(alcada_input)
    except ValueError:
        print("Advertència: introdueix valors numèrics.")
        return None

    es_valid, missatge = validar_dades(pes, alcada)
    if not es_valid:
        print(f"Advertència: {missatge}")
        return None

    imc = calcular_imc(pes, alcada)
    categoria = classificar_imc(imc)
    return {"imc": round(imc, 2), "categoria": categoria}


resultado = imc_usuari()
print(resultado)

Advertència: Valors fora de rang raonable (pes > 300 kg o alçada > 2.5 m).
None


Demostració amb valors fixos per reproduir el resultat:

In [4]:
pes, alcada = 70.0, 1.72
imc = calcular_imc(pes, alcada)
categoria = classificar_imc(imc)
{"imc": round(imc, 2), "categoria": categoria}

{'imc': 23.66, 'categoria': 'Pes normal'}

### Exercici 2 — Convertidor de temperatures

La funció permet convertir temperatures entre Celsius, Fahrenheit i Kelvin. Totes les conversions possibles s'emmagatzemen en un únic diccionari amb claus de tipus tupla `(origen, destí)`.

In [5]:
CONVERSIONS = {
    ("C", "F"): lambda c: c * 9 / 5 + 32,
    ("C", "K"): lambda c: c + 273.15,
    ("F", "C"): lambda f: (f - 32) * 5 / 9,
    ("F", "K"): lambda f: (f - 32) * 5 / 9 + 273.15,
    ("K", "C"): lambda k: k - 273.15,
    ("K", "F"): lambda k: (k - 273.15) * 9 / 5 + 32,
}

UNITATS_VALIDES = {"C", "F", "K"}


def convertir_temperatura(valor, origen):

    origen = origen.upper()
    if origen not in UNITATS_VALIDES:
        return f"Advertència: unitat no vàlida '{origen}'. Utilitza C, F o K."
    try:
        valor = float(valor)
    except (ValueError, TypeError):
        return "Advertència: el valor ha de ser numèric."

    return {
        desti: round(func(valor), 4)
        for (orig, desti), func in CONVERSIONS.items()
        if orig == origen
    }

In [6]:
convertir_temperatura(25, "C")

{'F': 77.0, 'K': 298.15}

In [7]:
convertir_temperatura(32, "F")

{'C': 0.0, 'K': 273.15}

#### Extra
 Pensa una manera d’emmagatzemar totes les possibles conversions en un sol objecte (Llista? Diccionari? DataFrame?) en comptes d’escriure molts if else en funció de la temperatura d’origen i la temperatura de destí.

Totes les conversions ja estan emmagatzemades en un sol diccionari (`CONVERSIONS`) indexat per tuples `(origen, destí)`. Això substitueix qualsevol cadena de `if/elif` i permet afegir noves unitats.

### Exercici 3 — Comptador de paraules d'un text

La funció normalitza el text (minúscules, eliminació de puntuació) i compta la freqüència de cada paraula. 

In [8]:
def netejar_text(text):
    text = text.lower()
    return re.sub(r"[^\w\s]", "", text)


def comptar_paraules(text):
    if not isinstance(text, str):
        return "Advertència: l'entrada ha de ser una cadena de text."

    paraules = netejar_text(text).split()

    if not paraules:
        return {}

    return dict(Counter(paraules))

#EXTRA
def longitud_mitjana(text):
    if not isinstance(text, str):
        return "Advertència: l'entrada ha de ser una cadena de text."

    paraules = netejar_text(text).split()

    if not paraules:
        return 0.0

    return sum(len(p) for p in paraules) / len(paraules)

In [9]:
comptar_paraules("Hola, hola! Com va?")

{'hola': 2, 'com': 1, 'va': 1}

#### Extra — Longitud mitjana de les paraules

In [10]:
longitud_mitjana("Hola com va?")

3.0

### Exercici 4 — Diccionari invers (amb possibilitat de duplicats)

La funció intercanvia les claus i valors d'un diccionari. Si hi ha valors repetits, els agrupa en una llista i emet un missatge d'advertència, tal com s'indica a l'enunciat.

In [11]:
def diccionari_invers(dic):

    if not isinstance(dic, dict):
        return "Advertència: l'entrada ha de ser un diccionari."

    resultat = {}

    for clau, valor in dic.items():
        if valor not in resultat:
            resultat[valor] = clau
        else:
            if not isinstance(resultat[valor], list):
                resultat[valor] = [resultat[valor]]
            resultat[valor].append(clau)

    duplicats = {v: k for v, k in resultat.items() if isinstance(k, list)}
    if duplicats:
        print(f"Advertència: valors duplicats trobats: {duplicats}")

    return resultat

In [12]:
diccionari_invers({'x': 'apple', 'y': 'banana', 'z': 'banana'})

Advertència: valors duplicats trobats: {'banana': ['y', 'z']}


{'apple': 'x', 'banana': ['y', 'z']}

---
## Nivell 2

### Exercici 1 — Comptador i endreçador de paraules d'un fitxer TXT


In [13]:
def analitzar_fitxer(ruta):
    with open(ruta, "r", encoding="utf-8") as f:
        text = f.read()

    paraules = netejar_text(text).split()
    conteo = Counter(paraules)

    resultat = {}
    for paraula, freq in conteo.items():
        lletra = paraula[0]
        if lletra not in resultat:
            resultat[lletra] = {}
        resultat[lletra][paraula] = freq

    return {
        lletra: dict(sorted(paraules.items()))
        for lletra, paraules in sorted(resultat.items())
    }

Per executar la funció cal tenir el fitxer al sistema. Aquí es mostra el resultat obtingut amb `tu_me_quieres_blanca.txt`:

In [14]:
# resultat = analitzar_fitxer("tu_me_quieres_blanca.txt")
# resultat

# Resultat precalculat per reproduïbilitat:
{'a': {'a': 3, 'agua': 1, 'al': 2, 'alba': 4, 'alcobas': 1, 'alimenta': 1, 'alma': 1, 'amarga': 1, 'azucena': 1}, 'b': {'baco': 1, 'banquete': 1, 'bebe': 1, 'blanca': 3, 'boca': 1, 'bosques': 1, 'buen': 1}, 'c': {'cabañas': 1, 'carnes': 2, 'casta': 3, 'cerrada': 1, 'con': 4, 'conservas': 1, 'copas': 1, 'corola': 1, 'corriste': 1, 'cuando': 2, 'cubierto': 1, 'cuerpo': 1, 'cuáles': 1}, 'd': {'de': 8, 'dejaste': 1, 'del': 1, 'diga': 1, 'dios': 2, 'duerme': 1}, 'e': {'el': 4, 'ellas': 1, 'en': 4, 'engaño': 1, 'enredada': 1, 'entonces': 1, 'escarcha': 1, 'espumas': 1, 'esqueleto': 1, 'estrago': 1}, 'f': {'festejando': 1, 'filtrado': 1, 'frutos': 1}, 'h': {'habla': 1, 'hacia': 1, 'haya': 1, 'hayas': 1, 'hermana': 1, 'hombre': 1, 'hubiste': 1, 'huye': 1}, 'i': {'intacto': 1}, 'j': {'jardines': 1}, 'l': {'la': 3, 'labios': 1, 'las': 7, 'lo': 2, 'los': 4, 'luna': 1, 'lévate': 1, 'límpiate': 1}, 'm': {'mano': 1, 'manos': 1, 'margarita': 1, 'me': 10, 'mi': 1, 'mieles': 1, 'milagros': 1, 'mojada': 1, 'montaña': 1, 'morados': 1}, 'n': {'negros': 1, 'ni': 2, 'no': 1, 'nácar': 1, 'nívea': 2}, 'p': {'perdone': 2, 'perfume': 1, 'por': 2, 'pretendes': 3, 'preténdeme': 3, 'puesto': 1, 'pájaros': 1, 'pámpanos': 1}, 'q': {'que': 6, 'quedó': 1, 'quieres': 6}, 'r': {'rayo': 1, 'raíz': 1, 'renueva': 1, 'rocas': 1, 'rojo': 1}, 's': {'salitre': 1, 'se': 2, 'sea': 1, 'sean': 1, 'sobre': 2, 'sé': 1}, 't': {'te': 3, 'tejidos': 1, 'tenue': 1, 'tierra': 1, 'toca': 1, 'todas': 2, 'todavía': 1, 'tornadas': 1, 'tú': 8}, 'u': {'un': 1, 'una': 1}, 'v': {'vestido': 1, 'vete': 1, 'vive': 1}, 'y': {'y': 5}}

{'a': {'a': 3,
  'agua': 1,
  'al': 2,
  'alba': 4,
  'alcobas': 1,
  'alimenta': 1,
  'alma': 1,
  'amarga': 1,
  'azucena': 1},
 'b': {'baco': 1,
  'banquete': 1,
  'bebe': 1,
  'blanca': 3,
  'boca': 1,
  'bosques': 1,
  'buen': 1},
 'c': {'cabañas': 1,
  'carnes': 2,
  'casta': 3,
  'cerrada': 1,
  'con': 4,
  'conservas': 1,
  'copas': 1,
  'corola': 1,
  'corriste': 1,
  'cuando': 2,
  'cubierto': 1,
  'cuerpo': 1,
  'cuáles': 1},
 'd': {'de': 8, 'dejaste': 1, 'del': 1, 'diga': 1, 'dios': 2, 'duerme': 1},
 'e': {'el': 4,
  'ellas': 1,
  'en': 4,
  'engaño': 1,
  'enredada': 1,
  'entonces': 1,
  'escarcha': 1,
  'espumas': 1,
  'esqueleto': 1,
  'estrago': 1},
 'f': {'festejando': 1, 'filtrado': 1, 'frutos': 1},
 'h': {'habla': 1,
  'hacia': 1,
  'haya': 1,
  'hayas': 1,
  'hermana': 1,
  'hombre': 1,
  'hubiste': 1,
  'huye': 1},
 'i': {'intacto': 1},
 'j': {'jardines': 1},
 'l': {'la': 3,
  'labios': 1,
  'las': 7,
  'lo': 2,
  'los': 4,
  'luna': 1,
  'lévate': 1,
  'límpiate'

### Exercici 2 — Conversió de tipus de dades

La funció recorre recursivament una llista que pot contenir llistes i tuples, i separa els elements que es poden convertir a `float` dels que no. Es serveix `try/except` de forma controlada, tal com s'indica a les bones pràctiques: únicament per gestionar casos previstos com ara elements no convertibles.

In [15]:
def conversio(dades):

    convertibles = []
    no_convertibles = []

    def processar(element):
        if isinstance(element, (list, tuple)):
            for item in element:
                processar(item)
        else:
            try:
                convertibles.append(float(element))
            except (ValueError, TypeError):
                no_convertibles.append(element)

    processar(dades)
    return convertibles, no_convertibles

In [16]:
dades = ['1.3', 'one', '1e10', 'seven', '3-1/2', ('2', 1, 1.4, 'not-a-number'), [1, 2, '3', '3.4']]
conversio(dades)

([1.3, 10000000000.0, 2.0, 1.0, 1.4, 1.0, 2.0, 3.0, 3.4],
 ['one', 'seven', '3-1/2', 'not-a-number'])

---
## Nivell 3

### Exercici 1 — Generador de contrasenyes

La funció genera contrasenyes aleatòries segures utilitzant `numpy.random`. Per garantir que es compleix cada criteri activat (majúscules, minúscules, números, signes), primer s'afegeix un caràcter mínim de cada grup i després s'omple la resta fins a la longitud desitjada; finalment es barreja el resultat.

Com a extra, `pyperclip` còpia la contrasenya automàticament al porta-retalls.

In [17]:
MAJUSCULES = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
MINUSCULES = "abcdefghijklmnopqrstuvwxyz"
NUMEROS = "0123456789"
SIGNES = ",-$?#!"

def crear_contrasenya(longitud, majuscules=True, minuscules=True, numeros=True, signes=False):
    grups = []
    if majuscules:
        grups.append(MAJUSCULES)
    if minuscules:
        grups.append(MINUSCULES)
    if numeros:
        grups.append(NUMEROS)
    if signes:
        grups.append(SIGNES)

    if not grups:
        return "Error: selecciona almenys un tipus de caràcter."
    if longitud < len(grups):
        return f"Error: la longitud mínima per als paràmetres escollits és {len(grups)}."

    # garantir almenys un caràcter de cada grup
    password = [np.random.choice(list(grup)) for grup in grups]

    tots = "".join(grups)
    password += list(np.random.choice(list(tots), size=longitud - len(password)))

    password = list(np.random.permutation(password))
    resultat = "".join(password)

    pyperclip.copy(resultat)
    return resultat

In [18]:
crear_contrasenya(10)

'EAT87jpEvi'

In [19]:
crear_contrasenya(10, True, True, True, True)

'M!x#t1W7Es'

### Exercici 2 — Processament de dades simple (partits de futbol)

La funció llegeix el fitxer `historic_partits.txt` i calcula per a cada equip: gols marcats, gols encaixats, partits guanyats/empatats/perduts i punts. A partir d'aquestes dades es poden obtenir l'equip més golejador, el més golejat i la classificació general.

In [26]:
def inicialitzar_equip():
    return {"gols": 0, "gols_encaixats": 0, "guanyats": 0, "empatats": 0, "perduts": 0, "punts": 0}


def actualitzar_resultat(equips, eq1, eq2, g1, g2):
    equips[eq1]["gols"] += g1
    equips[eq1]["gols_encaixats"] += g2
    equips[eq2]["gols"] += g2
    equips[eq2]["gols_encaixats"] += g1

    if g1 > g2:
        equips[eq1]["guanyats"] += 1
        equips[eq1]["punts"] += 3
        equips[eq2]["perduts"] += 1
    elif g1 < g2:
        equips[eq2]["guanyats"] += 1
        equips[eq2]["punts"] += 3
        equips[eq1]["perduts"] += 1
    else:
        equips[eq1]["empatats"] += 1
        equips[eq1]["punts"] += 1
        equips[eq2]["empatats"] += 1
        equips[eq2]["punts"] += 1


def processar_partits(ruta):
    equips = {}

    with open(ruta, "r", encoding="utf-8") as f:
        for linia in f:
            equip1, resultat, equip2 = linia.strip().split("\t")
            gols1, gols2 = map(int, resultat.split("-"))

            for equip in [equip1, equip2]:
                if equip not in equips:
                    equips[equip] = inicialitzar_equip()
            actualitzar_resultat(equips, equip1, equip2, gols1, gols2)
    return equips


def equip_mes_golejador(equips):

    return max(equips, key=lambda e: equips[e]["gols"])


def equip_mes_golejat(equips):
    return max(equips, key=lambda e: equips[e]["gols_encaixats"])


def classificacio(equips):
    return sorted(equips.items(), key=lambda x: x[1]["punts"], reverse=True)

In [24]:
dades = processar_partits("historic_partits.txt")
equip_mes_golejador(dades)

'Figueres'

In [25]:
equip_mes_golejat(dades)

'Vilafranca'

In [23]:
classificacio(dades)[:5]

[('Girona FC',
  {'gols': 139,
   'gols_encaixats': 94,
   'guanyats': 31,
   'empatats': 3,
   'perduts': 13,
   'punts': 96}),
 ('Llagostera',
  {'gols': 159,
   'gols_encaixats': 142,
   'guanyats': 29,
   'empatats': 7,
   'perduts': 20,
   'punts': 94}),
 ('Sabadell',
  {'gols': 141,
   'gols_encaixats': 121,
   'guanyats': 26,
   'empatats': 7,
   'perduts': 15,
   'punts': 85}),
 ('Cornellà',
  {'gols': 147,
   'gols_encaixats': 146,
   'guanyats': 25,
   'empatats': 7,
   'perduts': 22,
   'punts': 82}),
 ('RCD Espanyol',
  {'gols': 131,
   'gols_encaixats': 144,
   'guanyats': 23,
   'empatats': 11,
   'perduts': 21,
   'punts': 80})]